# Fast-DetectGPT Single Threshold - Colab Notebook

Self-contained notebook for SemEval 2026 Task A. It calibrates one Fast-DetectGPT threshold on 2000 balanced train rows and evaluates Macro F1 on the first 1000 test rows for the selected models.

## 1. Install Dependencies

In [1]:
import os
import sys
import subprocess

IS_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
print(f"Running on {'Google Colab' if IS_COLAB else 'local Python'}")

packages = [
    "accelerate",
    "bitsandbytes",
    "datasets==4.3.0",
    "huggingface_hub>=0.34.0",
    "numpy",
    "pandas",
    "pyarrow",
    "scikit-learn",
    "tqdm",
    "transformers",
]

if IS_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("If imports fail locally, run: python -m pip install " + " ".join(packages))

Running on Google Colab


## 2. Configuration

In [2]:
from pathlib import Path

DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
DATASET_CONFIG = "A"

MODEL_ALIASES = {
    "CodeLlama-7b-Instruct-hf": "codellama/CodeLlama-7b-Instruct-hf",
    "Starcoder2-3b": "bigcode/starcoder2-3b",
    "deepseek-coder-1.3b-base": "deepseek-ai/deepseek-coder-1.3b-base",
    "deepseek-coder-6.7b-base": "deepseek-ai/deepseek-coder-6.7b-base",
    "Starcoder2-7b": "bigcode/starcoder2-7b",
    "Qwen2.5-Coder-3B": "Qwen/Qwen2.5-Coder-3B",
    "CodeLlama-7b-hf": "codellama/CodeLlama-7b-hf",
    "Qwen2.5-Coder-1.5B": "Qwen/Qwen2.5-Coder-1.5B",
    "falcon-7b": "tiiuae/falcon-7b",
    "Yi-Coder-9B": "01-ai/Yi-Coder-9B",
    "Qwen2.5-Coder-7B-Instruct": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen2.5-Coder-3B-Instruct": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "deepseek-coder-1.3b-instruct": "deepseek-ai/deepseek-coder-1.3b-instruct",
    "Qwen2.5-Coder-7B": "Qwen/Qwen2.5-Coder-7B",
    "CodeLlama-7b-Python-hf": "codellama/CodeLlama-7b-Python-hf",
    "deepseek-coder-6.7b-instruct": "deepseek-ai/deepseek-coder-6.7b-instruct",
    "Yi-Coder-9B-Chat": "01-ai/Yi-Coder-9B-Chat",
    "CodeGemma-7b": "google/codegemma-7b",
    "CodeGemma-7b-it": "google/codegemma-7b-it",
    "stable-code-3b": "stabilityai/stable-code-3b",
}

MODELS = list(MODEL_ALIASES.keys())

TRAIN_THRESHOLD_SAMPLES = 2000
TEST_SAMPLES = 1000
THRESHOLD_GRID_SIZE = 200
MAX_LENGTH = 512
SEED = 42

# Keep this True on Colab T4/L4 to reduce VRAM use. Set False if you want full fp16 loading.
USE_8BIT_ON_CUDA = True

# Set True if you want to recompute scores even when cached CSV files exist.
FORCE_RESCORE = False

OUTPUT_DIR = Path("results")
CACHE_DIR = OUTPUT_DIR / "cache"
REPORTS_DIR = OUTPUT_DIR / "reports"
OUTPUT_CSV = OUTPUT_DIR / "result.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Models to evaluate: {len(MODELS)}")
print(f"Output CSV: {OUTPUT_CSV}")

Models to evaluate: 20
Output CSV: results/result.csv


## 3. Imports, Seed, Device

In [3]:
import gc
import json
import random

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import classification_report, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Device: cuda
NVIDIA A100-SXM4-40GB
VRAM allocated: 0.00 GB


## 4. Load Task A Data

In [4]:
print(f"Loading dataset {DATASET_NAME}, config {DATASET_CONFIG}...")
train_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
test_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

train_df = train_dataset.to_pandas().dropna(subset=["code", "label"]).copy()
test_df = test_dataset.to_pandas().dropna(subset=["code"]).head(TEST_SAMPLES).copy()

train_df["label"] = train_df["label"].astype(int)
if "label" not in test_df.columns:
    raise ValueError("The test split has no label column, so Macro F1 cannot be computed.")
test_df["label"] = test_df["label"].astype(int)

per_class = TRAIN_THRESHOLD_SAMPLES // 2
threshold_df = pd.concat([
    train_df[train_df["label"] == 0].head(per_class),
    train_df[train_df["label"] == 1].head(per_class),
]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Threshold calibration rows:")
print(threshold_df["label"].value_counts().sort_index())
print("Test rows:")
print(test_df["label"].value_counts().sort_index())

Loading dataset DaniilOr/SemEval-2026-Task13, config A...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Threshold calibration rows:
label
0    1000
1    1000
Name: count, dtype: int64
Test rows:
label
0    777
1    223
Name: count, dtype: int64


## 5. Fast-DetectGPT Single-Threshold Functions

In [5]:
def safe_model_key(model_name):
    return model_name.replace("/", "__").replace(" ", "_")


def get_sampling_discrepancy_analytic(logits_ref, logits_score, labels):
    assert logits_ref.shape[0] == 1
    assert logits_score.shape[0] == 1
    assert labels.shape[0] == 1

    if logits_ref.size(-1) != logits_score.size(-1):
        vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
        logits_ref = logits_ref[:, :, :vocab_size]
        logits_score = logits_score[:, :, :vocab_size]

    labels = labels.unsqueeze(-1) if labels.ndim == logits_score.ndim - 1 else labels
    lprobs_score = torch.log_softmax(logits_score, dim=-1)
    probs_ref = torch.softmax(logits_ref, dim=-1)

    log_likelihood = lprobs_score.gather(dim=-1, index=labels).squeeze(-1)
    mean_ref = (probs_ref * lprobs_score).sum(dim=-1)
    var_ref = (probs_ref * torch.square(lprobs_score)).sum(dim=-1) - torch.square(mean_ref)

    discrepancy = (log_likelihood.sum(dim=-1) - mean_ref.sum(dim=-1)) / (var_ref.sum(dim=-1).sqrt() + 1e-8)
    return discrepancy.mean().item()


def fit_single_threshold(y_true, y_scores, grid_size):
    thresholds = np.linspace(np.min(y_scores), np.max(y_scores), grid_size)
    best_threshold = float(thresholds[0])
    best_f1 = -1.0

    for threshold in thresholds:
        preds = (y_scores > threshold).astype(int)
        f1 = f1_score(y_true, preds, average="macro")
        if f1 > best_f1:
            best_f1 = float(f1)
            best_threshold = float(threshold)

    return best_threshold, best_f1


def load_model_and_tokenizer(model_id):
    print(f"Loading tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"trust_remote_code": True}
    if DEVICE == "cuda":
        model_kwargs["device_map"] = "auto"
        model_kwargs["torch_dtype"] = torch.float16
        if USE_8BIT_ON_CUDA and BitsAndBytesConfig is not None:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    else:
        model_kwargs["torch_dtype"] = torch.float32
        model_kwargs["low_cpu_mem_usage"] = True

    print(f"Loading model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    if DEVICE == "cpu":
        model.to(DEVICE)
    model.eval()
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return tokenizer, model


def compute_scores(df, tokenizer, model, desc):
    scores = []

    for row in tqdm(df.to_dict("records"), total=len(df), desc=desc):
        tokenized = tokenizer(
            row["code"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            return_token_type_ids=False,
        ).to(DEVICE)

        if tokenized.input_ids.shape[1] <= 5:
            scores.append(0.0)
            continue

        labels = tokenized.input_ids[:, 1:]
        with torch.no_grad():
            try:
                logits = model(**tokenized).logits[:, :-1]
                score = get_sampling_discrepancy_analytic(logits, logits, labels)
            except Exception as exc:
                print(f"Warning: scoring failed for one row: {exc}")
                score = 0.0
        scores.append(score)

    scored_df = df[["label"]].copy()
    scored_df["discrepancy_score"] = scores
    return scored_df


def cleanup_model(tokenizer=None, model=None, model_id=None):
    """Free GPU/CPU memory AND purge the HF cache for this model to reclaim disk space."""
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Purge HF hub cache for this model to free disk space on Colab
    if model_id is not None:
        import shutil
        hf_cache = Path.home() / ".cache" / "huggingface" / "hub"
        model_cache_name = "models--" + model_id.replace("/", "--")
        model_cache_path = hf_cache / model_cache_name
        if model_cache_path.exists():
            print(f"Purging HF cache: {model_cache_path}")
            shutil.rmtree(model_cache_path, ignore_errors=True)

## 6. Evaluate One Model

In [ ]:
def load_or_compute_scores(model_label, model_id):
    model_cache_dir = CACHE_DIR / safe_model_key(model_label)
    train_scores_path = model_cache_dir / "train_threshold_scores.csv"
    test_scores_path = model_cache_dir / "test_scores.csv"

    if not FORCE_RESCORE and train_scores_path.exists() and test_scores_path.exists():
        print(f"Using cached scores for {model_label}")
        return pd.read_csv(train_scores_path), pd.read_csv(test_scores_path)

    model_cache_dir.mkdir(parents=True, exist_ok=True)
    tokenizer, model = load_model_and_tokenizer(model_id)
    try:
        train_scores_df = compute_scores(
            threshold_df,
            tokenizer,
            model,
            desc=f"{model_label}: threshold rows",
        )
        test_scores_df = compute_scores(
            test_df,
            tokenizer,
            model,
            desc=f"{model_label}: test rows",
        )
        train_scores_df.to_csv(train_scores_path, index=False)
        test_scores_df.to_csv(test_scores_path, index=False)
        return train_scores_df, test_scores_df
    finally:
        cleanup_model(tokenizer, model, model_id=model_id)


def evaluate_model(model_label):
    model_id = MODEL_ALIASES.get(model_label, model_label)
    train_scores_df, test_scores_df = load_or_compute_scores(model_label, model_id)

    train_y = train_scores_df["label"].astype(int).to_numpy()
    train_scores = train_scores_df["discrepancy_score"].to_numpy()
    threshold, train_f1 = fit_single_threshold(train_y, train_scores, THRESHOLD_GRID_SIZE)

    test_y = test_scores_df["label"].astype(int).to_numpy()
    test_scores = test_scores_df["discrepancy_score"].to_numpy()
    test_preds = (test_scores > threshold).astype(int)
    test_f1 = float(f1_score(test_y, test_preds, average="macro"))

    report = classification_report(
        test_y,
        test_preds,
        labels=[0, 1],
        target_names=["human", "ai"],
        output_dict=True,
        zero_division=0,
    )

    report_path = REPORTS_DIR / f"{safe_model_key(model_label)}.json"
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    result = {
        "model": model_label,
        "model_id": model_id,
        "threshold": threshold,
        "train_macro_f1": train_f1,
        "test_macro_f1": test_f1,
        "test_human_f1": float(report["human"]["f1-score"]),
        "test_ai_f1": float(report["ai"]["f1-score"]),
        "train_rows": int(len(train_scores_df)),
        "test_rows": int(len(test_scores_df)),
        "report_path": str(report_path),
        "error": "",
    }

    print(
        f"{model_label}: threshold={threshold:.4f}, "
        f"train_macro_f1={train_f1:.4f}, test_macro_f1={test_f1:.4f}"
    )
    return result

## 7. Run All Selected Models

In [7]:
results = []
if OUTPUT_CSV.exists() and not FORCE_RESCORE:
    results = pd.read_csv(OUTPUT_CSV).to_dict("records")

completed = {
    row.get("model")
    for row in results
    if row.get("model") and not row.get("error") and pd.notna(row.get("test_macro_f1"))
}

for model_label in MODELS:
    if model_label in completed:
        print(f"Skipping completed model: {model_label}")
        continue

    try:
        result = evaluate_model(model_label)
    except Exception as exc:
        cleanup_model(model_id=MODEL_ALIASES.get(model_label, model_label))
        result = {
            "model": model_label,
            "model_id": MODEL_ALIASES.get(model_label, model_label),
            "threshold": np.nan,
            "train_macro_f1": np.nan,
            "test_macro_f1": np.nan,
            "test_human_f1": np.nan,
            "test_ai_f1": np.nan,
            "train_rows": 0,
            "test_rows": 0,
            "report_path": "",
            "error": repr(exc),
        }
        print(f"{model_label}: failed with {exc!r}")

    results.append(result)
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    print(f"Saved partial results to {OUTPUT_CSV}")

result_df = pd.DataFrame(results)
result_df.sort_values("test_macro_f1", ascending=False, na_position="last")

Loading tokenizer: codellama/CodeLlama-7b-Instruct-hf


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

stable-code-3b: threshold rows:   0%|          | 0/2000 [00:00<?, ?it/s]

stable-code-3b: test rows:   0%|          | 0/1000 [00:00<?, ?it/s]

Purging HF cache: /root/.cache/huggingface/hub/models--stabilityai--stable-code-3b
stable-code-3b: threshold=1.0671, train_macro_f1=0.5500, test_macro_f1=0.7219
Saved partial results to results/result.csv


,model,model_id,threshold,train_macro_f1,test_macro_f1,test_human_f1,test_ai_f1,train_rows,test_rows,report_path,error
19,stable-code-3b,stabilityai/stable-code-3b,1.067054,0.550049,0.721869,0.889582,0.554156,2000,1000,results/reports/stable-code-3b.json,
4,Starcoder2-7b,bigcode/starcoder2-7b,1.309399,0.538881,0.710278,0.893382,0.527174,2000,1000,results/reports/Starcoder2-7b.json,
6,CodeLlama-7b-hf,codellama/CodeLlama-7b-hf,0.494543,0.559826,0.703000,0.857143,0.548857,2000,1000,results/reports/CodeLlama-7b-hf.json,
1,Starcoder2-3b,bigcode/starcoder2-3b,1.478368,0.539368,0.697471,0.890787,0.504155,2000,1000,results/reports/Starcoder2-3b.json,
0,CodeLlama-7b-Instruct-hf,codellama/CodeLlama-7b-Instruct-hf,0.378671,0.549027,0.675160,0.852523,0.497797,2000,1000,results/reports/CodeLlama-7b-Instruct-hf.json,
14,CodeLlama-7b-Python-hf,codellama/CodeLlama-7b-Python-hf,0.446392,0.570540,0.652632,0.818674,0.486590,2000,1000,results/reports/CodeLlama-7b-Python-hf.json,
5,Qwen2.5-Coder-3B,Qwen/Qwen2.5-Coder-3B,0.517823,0.501600,0.635656,0.828758,0.442553,2000,1000,results/reports/Qwen2.5-Coder-3B.json,
13,Qwen2.5-Coder-7B,Qwen/Qwen2.5-Coder-7B,0.292321,0.502078,0.603334,0.786594,0.420074,2000,1000,results/reports/Qwen2.5-Coder-7B.json,
7,Qwen2.5-Coder-1.5B,Qwen/Qwen2.5-Coder-1.5B,0.422837,0.496437,0.599985,0.790792,0.409178,2000,1000,results/reports/Qwen2.5-Coder-1.5B.json,
16,Yi-Coder-9B-Chat,01-ai/Yi-Coder-9B-Chat,-2.615038,0.468186,0.581538,0.843077,0.320000,2000,1000,results/reports/Yi-Coder-9B-Chat.json,


## 8. Show And Download Results

In [8]:
result_df = pd.read_csv(OUTPUT_CSV)
display_cols = ["model", "threshold", "train_macro_f1", "test_macro_f1", "test_human_f1", "test_ai_f1", "error"]
display(result_df.reindex(columns=display_cols).sort_values("test_macro_f1", ascending=False, na_position="last"))

if IS_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_CSV))

,model,threshold,train_macro_f1,test_macro_f1,test_human_f1,test_ai_f1,error
19,stable-code-3b,1.067054,0.550049,0.721869,0.889582,0.554156,NaN
4,Starcoder2-7b,1.309399,0.538881,0.710278,0.893382,0.527174,NaN
6,CodeLlama-7b-hf,0.494543,0.559826,0.703000,0.857143,0.548857,NaN
1,Starcoder2-3b,1.478368,0.539368,0.697471,0.890787,0.504155,NaN
0,CodeLlama-7b-Instruct-hf,0.378671,0.549027,0.675160,0.852523,0.497797,NaN
14,CodeLlama-7b-Python-hf,0.446392,0.570540,0.652632,0.818674,0.486590,NaN
5,Qwen2.5-Coder-3B,0.517823,0.501600,0.635656,0.828758,0.442553,NaN
13,Qwen2.5-Coder-7B,0.292321,0.502078,0.603334,0.786594,0.420074,NaN
7,Qwen2.5-Coder-1.5B,0.422837,0.496437,0.599985,0.790792,0.409178,NaN
16,Yi-Coder-9B-Chat,-2.615038,0.468186,0.581538,0.843077,0.320000,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>